# Data Exploration
Before start working on ETL pipeline, I want first to understand what data I am dealing with, see any anomalies, and figure out what is needed before creating a formal pipeline.

In [78]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F

from pathlib import Path

In [79]:
import os

# Spark 4.2 is built for Java 17. The java on PATH is the Java 8 browser plugin.
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"

spark = (
    SparkSession.builder.appName("DataExploration").
    master("local[*]").
    config("spark.sql.shuffle.partitions", 8).
    config("spark.sql.ansi.enabled", "true").
    getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


In [80]:
RAW_DATA_PATH = Path("data/raw")

In [81]:
def read_raw_data(name: str) -> DataFrame:
    """
    Read raw data from CSV file.
    """
    df = (spark.read.option("header", True)
         .option("inferSchema", True)
        .option("encoding", "utf-8")
        .csv(f"{RAW_DATA_PATH}/{name}.csv"))

    return df.toDF(*[c.replace("ufeff", "").strip() for c in df.columns])

In [82]:
customers = read_raw_data("customers").cache()
products = read_raw_data("products").cache()
orders = read_raw_data("orders").cache()

26/10/06 19:05:41 WARN CacheManager: Asked to cache already cached data.
26/10/06 19:05:41 WARN CacheManager: Asked to cache already cached data.
26/10/06 19:05:41 WARN CacheManager: Asked to cache already cached data.


In [83]:
for name, df in [("customers", customers), ("products", products), ("orders", orders)]:
    print(f"{name:10s} rows={df.count():>8,}  cols={df.columns}")

customers  rows=   4,389  cols=['CustomerID', 'Country']
products   rows=   5,635  cols=['StockCode', 'Description', 'UnitPrice']
orders     rows= 541,909  cols=['InvoiceNo', 'StockCode', 'Quantity', 'InvoiceDate', 'CustomerID']


In [84]:
customers.show(5)
products.show(5)
orders.show(5)

+----------+--------------+
|CustomerID|       Country|
+----------+--------------+
|     12346|United Kingdom|
|     12347|       Iceland|
|     12348|       Finland|
|     12349|         Italy|
|     12350|        Norway|
+----------+--------------+
only showing top 5 rows
+---------+--------------------+---------+
|StockCode|         Description|UnitPrice|
+---------+--------------------+---------+
|    10002|INFLATABLE POLITI...|     3.25|
|    10002|                NULL|     2.73|
|    10080|GROOVY CACTUS INF...|      4.0|
|    10080|                NULL|     3.24|
|    10080|               check|     0.12|
+---------+--------------------+---------+
only showing top 5 rows
+---------+---------+--------+---------------+----------+
|InvoiceNo|StockCode|Quantity|    InvoiceDate|CustomerID|
+---------+---------+--------+---------------+----------+
|   536370|    10002|      48| 12/1/2010 8:45|     12583|
|   536382|    10002|      12| 12/1/2010 9:45|     16098|
|   536756|    10002|  

In [85]:
customers.printSchema()
products.printSchema()
orders.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

root
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- UnitPrice: double (nullable = true)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- CustomerID: integer (nullable = true)



In [86]:
def null_profile(df):
    """How many nulls, blank strings, and distinct values each column has."""
    total = df.count()
    rows = []

    for column in df.columns:
        nulls = df.filter(F.col(column).isNull()).count()

        # Blank strings only make sense for text. Numbers just get 0.
        blanks = 0
        if dict(df.dtypes)[column] == "string":
            blanks = df.filter(F.trim(F.col(column)) == "").count()

        distinct = df.select(column).distinct().count()
        null_pct = round(100 * nulls / total, 2)

        rows.append((column, total, nulls, blanks, distinct, null_pct))

    return spark.createDataFrame(
        rows, ["column", "rows", "nulls", "blanks", "distinct", "null_pct"]
    )

In [87]:
def duplicates_report(df: DataFrame, keys: list, label: str) -> None:
    """How many keys appear more than once, and how many rows are exact duplicates."""
    duplicated_keys = df.groupBy(*keys).count().filter("count > 1")
    print(f"[{label}] keys {keys} appearing >1 time: {duplicated_keys.count():,}")
    print(f"[{label}] exact duplicate rows: {df.count() - df.distinct().count():,}")

## Customers

In [88]:
null_profile(customers).show()
duplicates_report(customers, ["CustomerID"], "customers")

+----------+----+-----+------+--------+--------+
|    column|rows|nulls|blanks|distinct|null_pct|
+----------+----+-----+------+--------+--------+
|CustomerID|4389|    9|     0|    4373|    0.21|
|   Country|4389|    0|     0|      38|     0.0|
+----------+----+-----+------+--------+--------+

[customers] keys ['CustomerID'] appearing >1 time: 9
[customers] exact duplicate rows: 0


In [89]:
# Customers with more than one country: which one is true?
conflicts = (customers.filter(F.col("CustomerID").isNotNull())
             .groupBy("CustomerID")
             .agg(F.collect_set("Country").alias("countries"))
             .filter(F.size("countries") > 1))
print("IDs with conflicting countries:", conflicts.count())
conflicts.show(truncate=False)

# Rows with null ID: are they usable at all?
customers.filter(F.col("CustomerID").isNull()).show()

IDs with conflicting countries: 8
+----------+------------------------+
|CustomerID|countries               |
+----------+------------------------+
|12370     |[Austria, Cyprus]       |
|12394     |[Belgium, Denmark]      |
|12417     |[Spain, Belgium]        |
|12422     |[Switzerland, Australia]|
|12429     |[Austria, Denmark]      |
|12431     |[Belgium, Australia]    |
|12455     |[Spain, Cyprus]         |
|12457     |[Switzerland, Cyprus]   |
+----------+------------------------+

+----------+--------------+
|CustomerID|       Country|
+----------+--------------+
|      NULL|United Kingdom|
|      NULL|   Unspecified|
|      NULL|     Hong Kong|
|      NULL|          EIRE|
|      NULL|        Israel|
|      NULL|      Portugal|
|      NULL|        France|
|      NULL|   Switzerland|
|      NULL|       Bahrain|
+----------+--------------+



In [90]:
# Full country value set: look for aliases and placeholders
(customers.groupBy("Country").count()
 .orderBy(F.desc("count"))
 .show(50, truncate=False))

+--------------------+-----+
|Country             |count|
+--------------------+-----+
|United Kingdom      |3951 |
|Germany             |95   |
|France              |88   |
|Spain               |31   |
|Belgium             |25   |
|Switzerland         |22   |
|Portugal            |20   |
|Italy               |15   |
|Finland             |12   |
|Austria             |11   |
|Norway              |10   |
|Australia           |9    |
|Channel Islands     |9    |
|Denmark             |9    |
|Netherlands         |9    |
|Cyprus              |8    |
|Sweden              |8    |
|Japan               |8    |
|Poland              |6    |
|Israel              |5    |
|Unspecified         |5    |
|EIRE                |4    |
|USA                 |4    |
|Greece              |4    |
|Canada              |4    |
|Bahrain             |3    |
|United Arab Emirates|2    |
|Malta               |2    |
|Iceland             |1    |
|RSA                 |1    |
|Czech Republic      |1    |
|Lithuania    

A few short names, like EIRE, RSA, USA
European Community is not a country
Additionaly: Unspecified, Channel Islands (?)

UK has most of the customers

# Products

In [91]:
null_profile(products).show()
duplicates_report(products, ["StockCode"], "products")

+-----------+----+-----+------+--------+--------+
|     column|rows|nulls|blanks|distinct|null_pct|
+-----------+----+-----+------+--------+--------+
|  StockCode|5635|    0|     0|    3965|     0.0|
|Description|5635|  960|     0|    4223|   17.04|
|  UnitPrice|5635|    0|     0|     501|     0.0|
+-----------+----+-----+------+--------+--------+

[products] keys ['StockCode'] appearing >1 time: 1,315
[products] exact duplicate rows: 0


In [92]:
per_code = products.groupBy("StockCode").agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("UnitPrice").alias("n_prices"),
    F.min(F.col("UnitPrice").cast("double")).alias("min_price"),
    F.max(F.col("UnitPrice").cast("double")).alias("max_price"),
)
per_code.groupBy("n_rows").count().orderBy("n_rows").show()
per_code.filter("n_prices > 1").orderBy(F.desc("n_rows")).show(10)

+------+-----+
|n_rows|count|
+------+-----+
|     1| 2650|
|     2| 1061|
|     3|  185|
|     4|   49|
|     5|   14|
|     6|    2|
|     7|    2|
|     8|    2|
+------+-----+

+---------+------+--------+---------+---------+
|StockCode|n_rows|n_prices|min_price|max_price|
+---------+------+--------+---------+---------+
|    20713|     8|       8|     0.06|     4.14|
|    23084|     8|       8|      0.6|     4.46|
|    21830|     7|       7|     0.06|     4.74|
|    85175|     7|       7|     1.85|     4.85|
|    23131|     6|       6|     1.05|     4.85|
|    21181|     6|       6|     0.14|     4.33|
|   72807A|     5|       5|     0.29|     4.52|
|    22121|     5|       5|     0.98|     3.73|
|    23343|     5|       5|     1.29|     4.84|
|    22812|     5|       5|     1.31|     4.47|
+---------+------+--------+---------+---------+
only showing top 10 rows


In [93]:
prices = products.select(F.col("UnitPrice").cast("double").alias("price"))
prices.summary().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|              5635|
|   mean| 2.524615794143745|
| stddev|1.4427835088183272|
|    min|               0.0|
|    25%|              1.28|
|    50%|              2.55|
|    75%|              3.79|
|    max|               5.0|
+-------+------------------+



In [94]:
# not numeric prices
products.filter(F.expr("try_cast(UnitPrice AS DOUBLE)").isNull()).count()

0

In [95]:
prices.filter("price = 0").count()

10

In [96]:
(products.filter(F.col("Description").isNotNull() & F.col("Description").rlike("[a-z]"))
 .groupBy(F.lower(F.trim("Description")).alias("desc")).count()
 .orderBy(F.desc("count")).show(30, truncate=False))

+----------------------------+-----+
|desc                        |count|
+----------------------------+-----+
|check                       |146  |
|damaged                     |57   |
|damages                     |44   |
|found                       |32   |
|sold as set on dotcom       |20   |
|adjustment                  |17   |
|amazon                      |12   |
|unsaleable, destroyed.      |9    |
|thrown away                 |9    |
|dotcom                      |7    |
|wet damaged                 |5    |
|damages?                    |5    |
|had been put aside          |5    |
|ebay                        |5    |
|smashed                     |4    |
|mailout                     |4    |
|missing                     |4    |
|crushed                     |3    |
|wet pallet                  |3    |
|incorrect stock entry.      |3    |
|rusty throw away            |2    |
|???missing                  |2    |
|sold as 1                   |2    |
|?missing                    |2    |
|

In [97]:
odd = products.filter(~F.upper(F.trim("StockCode")).rlike(r"^\d{5}[A-Z]*$"))
print(odd.count())
odd.select("StockCode", "Description", "UnitPrice").orderBy("StockCode").show(40, truncate=False)

41
+------------+-----------------------------------+---------+
|StockCode   |Description                        |UnitPrice|
+------------+-----------------------------------+---------+
|AMAZONFEE   |AMAZON FEE                         |4.09     |
|B           |Adjust bad debt                    |2.83     |
|BANK CHARGES|Bank Charges                       |3.8      |
|C2          |CARRIAGE                           |3.89     |
|C2          |NULL                               |0.68     |
|CRUK        |CRUK Commission                    |0.89     |
|D           |Discount                           |1.24     |
|DCGS0003    |BOXED GLASS ASHTRAY                |2.61     |
|DCGS0003    |ebay                               |4.02     |
|DCGS0004    |HAYNES CAMPER SHOULDER BAG         |3.1      |
|DCGS0055    |NULL                               |4.47     |
|DCGS0057    |NULL                               |4.14     |
|DCGS0066P   |NULL                               |3.92     |
|DCGS0067    |ebay   

# Orders

In [98]:
null_profile(orders).show()
duplicates_report(orders, ["InvoiceNo", "StockCode"], "orders")

+-----------+------+------+------+--------+--------+
|     column|  rows| nulls|blanks|distinct|null_pct|
+-----------+------+------+------+--------+--------+
|  InvoiceNo|541909|     0|     0|   25900|     0.0|
|  StockCode|541909|     0|     0|    4070|     0.0|
|   Quantity|541909|     0|     0|     722|     0.0|
|InvoiceDate|541909|     0|     0|   23262|     0.0|
| CustomerID|541909|135080|     0|    4373|   24.93|
+-----------+------+------+------+--------+--------+

[orders] keys ['InvoiceNo', 'StockCode'] appearing >1 time: 9,694
[orders] exact duplicate rows: 5,429


In [99]:
# Invoice types by prefix
(orders.withColumn("prefix", F.regexp_extract("InvoiceNo", r"^([A-Za-z]*)", 1))
 .groupBy("prefix").count().show())

+------+------+
|prefix| count|
+------+------+
|     C|  9288|
|      |532618|
|     A|     3|
+------+------+



In [100]:
q = orders.withColumn("qty", F.expr("try_cast(Quantity AS INT)"))
q.select("qty").summary().show()

+-------+------------------+
|summary|               qty|
+-------+------------------+
|  count|            541909|
|   mean|  9.55224954743324|
| stddev|218.08115785023438|
|    min|            -80995|
|    25%|                 1|
|    50%|                 3|
|    75%|                10|
|    max|             80995|
+-------+------------------+



min and max matches, need investigation what are they

In [101]:
q.filter(F.col("qty") == -80995).show()

+---------+---------+--------+--------------+----------+------+
|InvoiceNo|StockCode|Quantity|   InvoiceDate|CustomerID|   qty|
+---------+---------+--------+--------------+----------+------+
|  C581484|    23843|  -80995|12/9/2011 9:27|     16446|-80995|
+---------+---------+--------+--------------+----------+------+



In [102]:
q.filter(F.col("qty") == 80995).show()

+---------+---------+--------+--------------+----------+-----+
|InvoiceNo|StockCode|Quantity|   InvoiceDate|CustomerID|  qty|
+---------+---------+--------+--------------+----------+-----+
|   581483|    23843|   80995|12/9/2011 9:15|     16446|80995|
+---------+---------+--------+--------------+----------+-----+



looks like a mistake, but not sure, i will try to find other examples with negative qty and check if next invoices match

In [103]:
q.filter(F.col("qty") < 0).show()

+---------+---------+--------+----------------+----------+-----+
|InvoiceNo|StockCode|Quantity|     InvoiceDate|CustomerID|  qty|
+---------+---------+--------+----------------+----------+-----+
|   551429|    10002|      -3| 4/28/2011 15:05|      NULL|   -3|
|  C558716|    10133|     -10|  7/1/2011 13:22|     17888|  -10|
|   576843|    10133|     -82|11/16/2011 15:32|      NULL|  -82|
|   549031|    10134|     -19|  4/5/2011 17:04|      NULL|  -19|
|  C571707|    10135|      -1|10/18/2011 15:33|     14056|   -1|
|  C537043|    11001|      -1| 12/5/2010 10:48|     15351|   -1|
|  C565044|    11001|    -180| 8/31/2011 17:02|     12931| -180|
|  C568412|    11001|      -1| 9/27/2011 10:54|     14456|   -1|
|  C569682|    11001|      -2| 10/5/2011 14:33|     15351|   -2|
|  C575257|    11001|      -1| 11/9/2011 11:59|     13623|   -1|
|  C562124|    15034|     -23|  8/2/2011 17:23|     13081|  -23|
|  C562582|    15034|     -18|  8/7/2011 13:53|     15640|  -18|
|  C568416|    15034|    

In [104]:
q.filter((F.col("qty") == -1200) & (F.col("CustomerID") == "14533")).show()
q.filter((F.col("qty") == 1200) & (F.col("CustomerID") == "14533")).show()


+---------+---------+--------+---------------+----------+-----+
|InvoiceNo|StockCode|Quantity|    InvoiceDate|CustomerID|  qty|
+---------+---------+--------+---------------+----------+-----+
|  C569552|    15034|   -1200|10/4/2011 17:44|     14533|-1200|
+---------+---------+--------+---------------+----------+-----+

+---------+---------+--------+---------------+----------+----+
|InvoiceNo|StockCode|Quantity|    InvoiceDate|CustomerID| qty|
+---------+---------+--------+---------------+----------+----+
|   569214|    15034|    1200|10/2/2011 12:22|     14533|1200|
+---------+---------+--------+---------------+----------+----+



In [105]:
q.groupBy(
    F.col("InvoiceNo").startswith("C").alias("is_cancel"),
    (F.col("qty") < 0).alias("negative_qty"),
).count().orderBy("is_cancel", "negative_qty").show()

+---------+------------+------+
|is_cancel|negative_qty| count|
+---------+------------+------+
|    false|       false|531285|
|    false|        true|  1336|
|     true|        true|  9288|
+---------+------------+------+



In [106]:
q.orderBy(F.desc(F.abs("qty"))).show(10)

+---------+---------+--------+----------------+----------+------+
|InvoiceNo|StockCode|Quantity|     InvoiceDate|CustomerID|   qty|
+---------+---------+--------+----------------+----------+------+
|   581483|    23843|   80995|  12/9/2011 9:15|     16446| 80995|
|  C581484|    23843|  -80995|  12/9/2011 9:27|     16446|-80995|
|   541431|    23166|   74215| 1/18/2011 10:01|     12346| 74215|
|  C541433|    23166|  -74215| 1/18/2011 10:17|     12346|-74215|
|   578841|    84826|   12540|11/25/2011 15:57|     13256| 12540|
|   556690|    23005|   -9600| 6/14/2011 10:37|      NULL| -9600|
|   556691|    23005|   -9600| 6/14/2011 10:37|      NULL| -9600|
|  C536757|    84347|   -9360| 12/2/2010 14:23|     15838| -9360|
|   556687|    23003|   -9058| 6/14/2011 10:36|      NULL| -9058|
|   542504|    37413|    5568| 1/28/2011 12:03|      NULL|  5568|
+---------+---------+--------+----------------+----------+------+
only showing top 10 rows


In [107]:
FMT = "M/d/yyyy H:mm"
dated = orders.withColumn("ts", F.try_to_timestamp(F.trim("InvoiceDate"), F.lit(FMT)))

print("unparseable dates:")
dated.filter("ts IS NULL").select("InvoiceNo", "InvoiceDate").show(truncate=False)
dated.agg(F.min("ts"), F.max("ts")).show()

unparseable dates:
+---------+---------------+
|InvoiceNo|InvoiceDate    |
+---------+---------------+
|540239   |15/5/2011 14:48|
|540798   |1/11/2011 12:90|
+---------+---------------+

+-------------------+-------------------+
|            min(ts)|            max(ts)|
+-------------------+-------------------+
|2010-12-01 08:26:00|2011-12-09 12:50:00|
+-------------------+-------------------+



In [108]:
(dated.groupBy(F.date_format("ts", "yyyy-MM").alias("month"))
 .agg(F.count("*").alias("lines"), F.countDistinct("InvoiceNo").alias("invoices"))
 .orderBy("month").show(20))

+-------+-----+--------+
|  month|lines|invoices|
+-------+-----+--------+
|   NULL|    2|       2|
|2010-12|42481|    2025|
|2011-01|35145|    1476|
|2011-02|27707|    1393|
|2011-03|36748|    1983|
|2011-04|29916|    1744|
|2011-05|37030|    2162|
|2011-06|36874|    2012|
|2011-07|39518|    1927|
|2011-08|35284|    1737|
|2011-09|50226|    2327|
|2011-10|60742|    2637|
|2011-11|84711|    3462|
|2011-12|25525|    1015|
+-------+-----+--------+



In [109]:
dated.groupBy(F.date_format("ts", "E").alias("weekday")).count().orderBy(F.desc("count")).show()
dated.groupBy(F.hour("ts").alias("hour")).count().orderBy("hour").show(24)

+-------+------+
|weekday| count|
+-------+------+
|    Thu|103857|
|    Tue|101807|
|    Mon| 95111|
|    Wed| 94564|
|    Fri| 82193|
|    Sun| 64375|
|   NULL|     2|
+-------+------+

+----+-----+
|hour|count|
+----+-----+
|NULL|    2|
|   6|   41|
|   7|  383|
|   8| 8909|
|   9|34332|
|  10|49037|
|  11|57674|
|  12|78708|
|  13|72259|
|  14|67470|
|  15|77519|
|  16|54516|
|  17|28509|
|  18| 7974|
|  19| 3705|
|  20|  871|
+----+-----+



no orders on Saturdays, and order time between 6 and 20

In [110]:

orphan_customers = (orders.filter(F.col("CustomerID").isNotNull())
               .join(customers, "CustomerID", "left_anti"))
orphan_customers.count() # order lines with unknown customer 

0

In [111]:
raw_orphans = orders.join(products.select("StockCode").distinct(), "StockCode", "left_anti")
raw_orphans.count() # order lines with unknown product (raw)

2041

In [112]:
raw_orphans.groupBy("StockCode").count().orderBy(F.desc("count")).show(10)

+---------+-----+
|StockCode|count|
+---------+-----+
|   72349B|  118|
|   47566b|  102|
|   15056n|   84|
|   84997d|   75|
|   84997b|   70|
|   84509a|   68|
|   85123a|   67|
|   84970s|   66|
|   84997c|   66|
|   84997a|   65|
+---------+-----+
only showing top 10 rows


In [113]:
norm = lambda df: df.withColumn("code", F.upper(F.trim("StockCode")))
fixed_orphans = norm(orders).join(norm(products).select("code").distinct(), "code", "left_anti")

In [114]:
fixed_orphans.count()

0

In [115]:
customers.join(orders, "CustomerID", "left_anti").count()

9

In [116]:
norm(products).join(norm(orders), "code", "left_anti").select("code").distinct().count()

0

In [117]:
price_bounds = norm(products).groupBy("code").agg(
    F.min(F.col("UnitPrice").cast("decimal(10,2)")).alias("p_min"),
    F.max(F.col("UnitPrice").cast("decimal(10,2)")).alias("p_max"))


In [118]:
price_bounds.show()

+-----+-----+-----+
| code|p_min|p_max|
+-----+-----+-----+
|11001| 4.25| 4.25|
|16015| 0.34| 1.14|
|16048| 4.49| 4.49|
|16052| 1.28| 2.65|
|16225| 0.94| 0.94|
|18007| 0.25| 0.94|
|20654| 4.89| 4.89|
|20658| 1.54| 4.73|
|20662| 1.28| 1.28|
|20675| 2.26| 2.26|
|20711| 3.08| 4.19|
|20717| 1.47| 1.47|
|20749| 2.68| 2.68|
|20752| 0.44| 4.95|
|20767| 4.61| 4.61|
|20796| 1.45| 1.45|
|20803| 1.44| 4.55|
|20820| 3.32| 3.88|
|20839| 1.04| 4.83|
|20860| 1.38| 4.90|
+-----+-----+-----+
only showing top 20 rows


In [119]:
sales = (norm(orders)
         .withColumn("qty", F.expr("try_cast(Quantity AS INT)"))
         .filter(~F.col("InvoiceNo").rlike("^[CA]") & (F.col("qty") > 0))
         .join(price_bounds, "code"))

In [120]:
sales.agg(
    F.sum(F.col("qty") * F.col("p_min")).alias("revenue_at_min_price"),
    F.sum(F.col("qty") * F.col("p_max")).alias("revenue_at_max_price"),
    F.round(100 * F.sum(F.when(F.col("p_min") != F.col("p_max"), 1).otherwise(0)) / F.count("*"), 1)
      .alias("pct_lines_with_ambiguous_price"),
).show()

+--------------------+--------------------+------------------------------+
|revenue_at_min_price|revenue_at_max_price|pct_lines_with_ambiguous_price|
+--------------------+--------------------+------------------------------+
|         12519705.45|         16060015.59|                          31.0|
+--------------------+--------------------+------------------------------+

